In [ ]:
# =============================================================================
# FUMD-AI Training & Postprocessing Workflow -- Step 1b: Combine & window multiple datasets (optional)
# =============================================================================
# Step:         1 of 5 (alternate input path -- optional)
# Summary:      Combine several dataset_labeled_w<W>.csv runs into one training set (shifting vehicle ids so they can't collide), then window/encode/scale them exactly like Step 1.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-training-postprocessing-workflow
# Version:      0.1.3
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# =============================================================================

# Step 1b — Combine & window multiple datasets (optional)

Part of the **FUMD-AI training & postprocessing workflow**. An alternate
entry point into the same pipeline as Step 1
(`step_1_load_and_window_dataset.ipynb`), for training on *several*
preprocessing-workflow runs combined instead of just one.

**Why you'd want this.** A model trained on one run already generalizes
well to other runs of the same simulated area (see README's "Validation
status" -- a model trained on `1200_1` alone scored 94.3%-94.6% on six
other unseen runs, actually slightly *higher* than its own held-out
split). The likely payoff from combining several runs isn't the headline
accuracy number, which is already strong -- it's the *migration/handover
windows* specifically, the harder minority class (steady-state rows
dominate any single run). Combining runs roughly multiplies how many real
handover examples the model sees to learn from.

**Why not just combine everything you have.** If every available run goes
into training, there's no genuinely unseen run left to check
generalization against afterwards -- Step 3's `CROSS_RUN_DATASET_PATHS`
check becomes meaningless once the model has already seen everything you'd
evaluate it against. Deliberately hold out at least one run (ideally more)
from `DATASET_PATHS` below, and point Step 3's `CROSS_RUN_DATASET_PATHS`
at those held-out runs afterwards -- that combination needs no changes to
Step 3 at all, it already accepts a plain list of `dataset_labeled_w<W>.csv`
paths.

**The vehicle-id problem this solves.** Every preprocessing-workflow run
numbers its own vehicles independently starting from 0 -- `veh_id=500` in
one run and `veh_id=500` in another are two unrelated vehicles that
happen to share an id. Naively concatenating multiple runs' CSVs would
make every `veh_id`-grouped operation downstream (`split_vehicles`,
`build_sequences`, ...) silently treat those as *one* continuous vehicle,
building sliding windows that jump across the boundary between two
unrelated trajectories -- corrupting part of the training data with no
error or warning. `data.load_combined_labeled_dataset` shifts each
source file's `veh_id` by a large, distinct offset before concatenating
so this can never happen (see that function's own docstring for the
details), and adds a `source_dataset` column for provenance.

**Everything else is identical to Step 1** -- same categorical encoding,
same by-vehicle train/validation split (see `data.split_vehicles`'s
docstring for why splitting by *window* instead would leak near-duplicate
samples), same sliding-window construction, same train-only-fitted
scaler -- and it writes the exact same `windows.npz`/`scaler.joblib`/
`label_encoders.joblib`/`run_manifest.json` shape Step 1 does, so Step 2
onward need no changes at all to consume a combined run's output.

**Input:** `DATASET_PATHS` -- a list of two or more `dataset_labeled_w<W>.csv`
paths (no bundled default: the bundled example is one small slice, there's
nothing meaningful to combine it with out of the box -- this parameter
must be set).

**Outputs**, written under `OUTPUT_DIR` (default `pipeline_run_combined`,
distinct from Step 1's default `pipeline_run` so the two don't collide if
you run both against the same repo checkout):
- `windows.npz`, `scaler.joblib`, `label_encoders.joblib` -- same shape as Step 1
- `run_manifest.json` -- same fields as Step 1, plus `dataset_paths` (the
  full list combined) and `id_offset`


In [ ]:
import os
import sys

import joblib
import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # `jupyter execute` sets the kernel's cwd to this notebook's OWN
                     # directory (see nbclient's cli.py), not the repo root -- normalize
                     # back so `sys.path.insert(0, "src")` and every relative default
                     # below resolve the same way whether this notebook is run standalone
                     # (jupyter execute notebooks/step_N...ipynb, e.g. via slurm/*.sbatch)
                     # or chained by run_pipeline.ipynb (whose own cwd is already the repo
                     # root, and which additionally sets cwd explicitly -- see
                     # fumd_training_workflow.notebook_runner.run_step).

sys.path.insert(0, "src")  # so `import fumd_training_workflow` finds src/fumd_training_workflow/
from fumd_training_workflow import data as d
from fumd_training_workflow.evaluate import infer_num_base_stations
from fumd_training_workflow.metrics_io import save_run_manifest

In [ ]:
# ---- Parameters ----
# This cell is tagged "parameters" so the notebook can also be executed
# headlessly (see notebook_runner.run_step), or edited by hand for a
# standalone `jupyter execute` / Slurm submission.

# Two or more dataset_labeled_w<W>.csv paths to combine into one training
# set -- e.g. every "_1" run from a set of otherwise-identical simulated
# scenarios, holding the "_2"/"_3" runs back for Step 3's
# CROSS_RUN_DATASET_PATHS (see this notebook's markdown cell above for
# why). No default -- there's nothing meaningful to combine the bundled
# single example slice with.
DATASET_PATHS = []

# Shift each source file's veh_id by i * ID_OFFSET (i = its position in
# DATASET_PATHS) before combining, so vehicles from different runs can
# never collide -- see data.load_combined_labeled_dataset's docstring.
# The default comfortably covers any realistic SUMO run's vehicle count;
# raise it only if a source file's own max veh_id is >= this value (the
# loader raises a clear error in that case rather than silently
# corrupting the combined ids).
ID_OFFSET = 1_000_000

# None -> data.DEFAULT_FEATURE_COLUMNS (everything except the lead/leakage
# and label-only columns -- see data.py's module docstring).
FEATURE_COLUMNS = None

SEQUENCE_LENGTH = 6          # seconds of history per window
FUTURE_STEPS = [1, 2, 3, 4, 5, 6, 7]  # seconds ahead to forecast
VAL_FRACTION = 0.2           # fraction of *vehicles* held out for validation
RANDOM_SEED = 42

OUTPUT_DIR = "pipeline_run_combined"  # where windows.npz / scaler / encoders / manifest are written

In [ ]:
# ---- Environment variable overrides (optional) ----
# `jupyter execute` has no -p/parameter-injection flag (unlike papermill), so
# each parameter above can also be set via an environment variable of the same
# name instead of hand-editing this cell, e.g.:
#   DATASET_PATHS='["../datasets/900_1/dataset_labeled_w3.csv","../datasets/1000_1/dataset_labeled_w3.csv"]' \
#   OUTPUT_DIR=/abs/path/pipeline_run_combined \
#   jupyter execute notebooks/step_1b_combine_and_window_datasets.ipynb
# See fumd_training_workflow.notebook_runner.env_override's docstring for exact
# value-parsing rules. Unset variables leave the parameters cell's own default above untouched.
from fumd_training_workflow.notebook_runner import env_override

DATASET_PATHS = env_override('DATASET_PATHS', DATASET_PATHS)
ID_OFFSET = env_override('ID_OFFSET', ID_OFFSET)
FEATURE_COLUMNS = env_override('FEATURE_COLUMNS', FEATURE_COLUMNS)
SEQUENCE_LENGTH = env_override('SEQUENCE_LENGTH', SEQUENCE_LENGTH)
FUTURE_STEPS = env_override('FUTURE_STEPS', FUTURE_STEPS)
VAL_FRACTION = env_override('VAL_FRACTION', VAL_FRACTION)
RANDOM_SEED = env_override('RANDOM_SEED', RANDOM_SEED)
OUTPUT_DIR = env_override('OUTPUT_DIR', OUTPUT_DIR)

## 1. Load, combine, and normalize

In [ ]:
feature_columns = FEATURE_COLUMNS or d.DEFAULT_FEATURE_COLUMNS

df = d.load_combined_labeled_dataset(DATASET_PATHS, id_offset=ID_OFFSET)
print("combined:", df.shape)
print(df.groupby("source_dataset")["veh_id"].agg(rows="count", vehicles="nunique"))

df = d.ensure_one_row_per_vehicle_second(df)
print("after resample-guard:", df.shape)

num_base_stations = infer_num_base_stations(df)
print("num_base_stations (derived from combined data):", num_base_stations)

## 2. Encode categoricals

Fit once, on the whole combined dataset before splitting -- `lane` is a per-map road-edge id, and every value that appears anywhere in *any* combined run should get a stable code (not just the ones that happen to land in the training split). Any `lane` value present in a held-out run but never seen here still transforms safely later, via `SafeLabelEncoder`'s reserved "unknown" code.

In [ ]:
label_encoders = d.fit_label_encoders(df)
df = d.apply_label_encoders(df, label_encoders)
for col, enc in label_encoders.items():
    print(f"{col}: {len(enc.classes_)} classes (+ 1 reserved 'unknown' code)")

## 3. Split by vehicle

Splits across the *combined* pool of vehicles from every source dataset -- not stratified per source, just a single random shuffle-and-split the same as Step 1's (`data.split_vehicles`, unchanged). With enough vehicles per source this balances out proportionally on its own (e.g. three ~900-1200-vehicle runs combined split roughly 80/20 within each source, not just overall) -- worth spot-checking the per-source vehicle counts printed below if you combine sources of very different sizes.

In [ ]:
train_df, val_df = d.split_vehicles(df, val_fraction=VAL_FRACTION, seed=RANDOM_SEED)
print("train vehicles by source:")
print(train_df.groupby("source_dataset")["veh_id"].nunique())
print("val vehicles by source:")
print(val_df.groupby("source_dataset")["veh_id"].nunique())

## 4. Build sliding windows

In [ ]:
X_train, y_train, extra_train = d.build_sequences(
    train_df, feature_columns=feature_columns, sequence_length=SEQUENCE_LENGTH, future_steps=FUTURE_STEPS,
)
X_val, y_val, extra_val = d.build_sequences(
    val_df, feature_columns=feature_columns, sequence_length=SEQUENCE_LENGTH, future_steps=FUTURE_STEPS,
)

## 5. Fit & apply the scaler (train-only)

In [ ]:
scaler = d.fit_scaler(X_train)
X_train_scaled = d.apply_scaler(scaler, X_train)
X_val_scaled = d.apply_scaler(scaler, X_val)  # transform only -- never re-fit on validation data

## 6. Save

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.savez_compressed(
    os.path.join(OUTPUT_DIR, "windows.npz"),
    X_train=X_train_scaled, y_train=y_train,
    X_val=X_val_scaled, y_val=y_val,
    migration_train=extra_train["migration"], destination_train=extra_train["destination"],
    migration_val=extra_val["migration"], destination_val=extra_val["destination"],
)
joblib.dump(scaler, os.path.join(OUTPUT_DIR, "scaler.joblib"))
joblib.dump(label_encoders, os.path.join(OUTPUT_DIR, "label_encoders.joblib"))

manifest = {
    "dataset_path": ", ".join(DATASET_PATHS),  # human-readable summary -- see "dataset_paths" for the real list
    "dataset_paths": DATASET_PATHS,
    "id_offset": ID_OFFSET,
    "feature_columns": feature_columns,
    "sequence_length": SEQUENCE_LENGTH,
    "future_steps": FUTURE_STEPS,
    "num_base_stations": num_base_stations,
    "val_fraction": VAL_FRACTION,
    "random_seed": RANDOM_SEED,
    "n_train_windows": int(X_train_scaled.shape[0]),
    "n_val_windows": int(X_val_scaled.shape[0]),
}
save_run_manifest(OUTPUT_DIR, manifest)

print(f"saved windows.npz, scaler.joblib, label_encoders.joblib, run_manifest.json -> {OUTPUT_DIR}")
print(f"combined from {len(DATASET_PATHS)} datasets: {DATASET_PATHS}")
print(f"train windows: {X_train_scaled.shape}, val windows: {X_val_scaled.shape}")